In [1]:
using Pkg; Pkg.activate("../")

  Activating project at `~/Documents/Research/loss/experiments/AtmosLossDesign`


In [16]:
import JLD2
import YAML
import ClimaCalibrate as CAL
using ScikitLearn
using LinearAlgebra
using Statistics

using PyCall
const pykernels = PyNULL()
function __init__()
    copy!(pykernels, pyimport("sklearn.gaussian_process.kernels"))
end
__init__()
import CalibrateEmulateSample as CES 
using CalibrateEmulateSample.Emulators 
using EnsembleKalmanProcesses.DataContainers
using CalibrateEmulateSample.EnsembleKalmanProcesses
using CalibrateEmulateSample.MarkovChainMonteCarlo
import EnsembleKalmanProcesses as EKP

gppackage = Emulators.SKLJL()

SKLJL()

In [4]:
# ekp_fpath = "/central/scratch/jschmitt/calibrations/exp2/iteration_008/eki_file.jld2"
# eki_obj = JLD2.load_object(ekp_fpath)
# prior = CAL.get_prior("/central/groups/esm/jschmitt/ClimaAtmos.jl/calibration/experiments/gcm_driven_scm/prior_prognostic_pi_entr_smooth_entr_detr_impl_0M_v1.toml")
ekp_fpath = "eki_file.jld2"
eki_obj = JLD2.load_object(ekp_fpath);
prior = CAL.get_prior("prior_prognostic_pi_entr_smooth_entr_detr_impl_0M_v1.toml")

train_iopairs = CES.Utilities.get_training_points(eki_obj, 1:7)
test_iopairs = CES.Utilities.get_training_points(eki_obj, 8:8)

PairedDataContainer{Float64}(DataContainer{Float64}([1.159546629726542 0.9263727107949428 … 1.1427075042011816 1.0048236946575786; -4.4762393501871145 -4.488294677527606 … -4.472522051258442 -4.520923767746996; … ; -1.3593243770804737 -1.3186942505120218 … -1.3444793357820064 -1.2150246017439885; -1.059917327616584 -1.0302964115602709 … -1.0152195324965363 -0.9231952765138505]), DataContainer{Float64}([-0.796408480677717 -0.7971416236769785 … -0.7973539547095516 -0.7976844699961039; -0.7959965119899984 -0.7967873721684974 … -0.7970688159008599 -0.7974351114193305; … ; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265]))

In [ ]:
function clean_iopairs_nans(iopairs::PairedDataContainer, index_max)
    inputs = get_inputs(iopairs)
    outputs = get_outputs(iopairs)
    
    n_samples_in = size(inputs, 2)
    
    # Identify columns in the output matrix that contain any NaNs.
    # `any` over `dims=1` checks each column. `vec` converts the resulting row matrix to a vector.
    nan_output_cols = vec(any(isnan.(outputs), dims=1))
    
    # Indices of columns to keep (where there are no NaNs)
    kept_indices = findall(.!nan_output_cols)
    
    # Filter inputs and outputs to retain only the "good" columns
    cleaned_inputs = inputs[:, kept_indices]
    cleaned_outputs = outputs[1:index_max, kept_indices]
    
    n_removed = n_samples_in - length(kept_indices)
    println("Removed $(n_removed) samples with NaNs out of $(n_samples_in) total.")
    
    return PairedDataContainer(cleaned_inputs, cleaned_outputs)
end

# clean train and test iopairs
index_max = 900
cleaned_train_iopairs = clean_iopairs_nans(train_iopairs, index_max)
cleaned_test_iopairs = clean_iopairs_nans(test_iopairs, index_max)

Removed 102 samples with NaNs out of 700 total.
Removed 0 samples with NaNs out of 100 total.


PairedDataContainer{Float64}(DataContainer{Float64}([1.159546629726542 0.9263727107949428 … 1.1427075042011816 1.0048236946575786; -4.4762393501871145 -4.488294677527606 … -4.472522051258442 -4.520923767746996; … ; -1.3593243770804737 -1.3186942505120218 … -1.3444793357820064 -1.2150246017439885; -1.059917327616584 -1.0302964115602709 … -1.0152195324965363 -0.9231952765138505]), DataContainer{Float64}([-0.796408480677717 -0.7971416236769785 … -0.7973539547095516 -0.7976844699961039; -0.7959965119899984 -0.7967873721684974 … -0.7970688159008599 -0.7974351114193305; … ; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265]))

In [5]:
# unconstrained_inputs = CES.Utilities.get_inputs(input_output_pairs)
# inputs = Emulators.transform_unconstrained_to_constrained(prior, unconstrained_inputs)
# size(inputs)

(21, 500)

In [12]:
extrema(get_outputs(cleaned_train_iopairs))

(-1.4954772715911533, 2.111030821155452)

In [19]:
# Get all diagonal entries from each observation's covariance matrix
full_cov_matrix = Diagonal(vcat([diag(hcat(obs.covs...)) for obs in eki_obj.observation_series.observations[1:3]]...))

# Create diagonal matrix from these entries
# full_cov_matrix = Diagonal(vcat([diag(hcat(obs.covs...)) for obs in eki_obj.observation_series.observations]...))
# full_cov_matrix = Diagonal(diag(eki_obj.observation_series.observations[1].covs[1]))


# gp_kernel = pykernels.RBF(length_scale = 2.0)
# gauss_proc = Emulators.GaussianProcess(gppackage, 
#                                     noise_learn = false,
#                                     kernel = gp_kernel) 

# emulator_gp = Emulator(gauss_proc, 
#                     cleaned_train_iopairs;
#                     obs_noise_cov = full_cov_matrix,
#                     normalize_inputs = false,
#                     retained_svd_frac = 0.90)

# # emulator_gp = Emulator(gauss_proc, 

# optimize_hyperparameters!(emulator_gp)
# mcmc = MCMCWrapper(RWMHSampling(), get_obs(eki_obj)[1:index_max], prior, emulator_gp; init_params = init_sample);


nugget = 1e-3
overrides = Dict(
    "verbose" => true,
    "scheduler" => DataMisfitController(terminate_at = 20.0),
    "cov_sample_multiplier" => 1.0,
    "n_iteration" => 5,
    "n_features_opt" => 40,
    "n_ensemble" => 40, #tune 
)

n_features = 100
n_params = 21
kernel_structure = SeparableKernel(LowRankFactor(1, nugget), OneDimFactor()) # tune
mlt = ScalarRandomFeatureInterface(
    n_features,
    n_params,
    kernel_structure = kernel_structure,
    optimizer_options = overrides,
)


emulator_rf = Emulator(mlt, 
                    cleaned_train_iopairs;
                    obs_noise_cov = full_cov_matrix,
                    normalize_inputs = true,
                    retained_svd_frac = 0.8)


SVD truncated at k: 184/270


┌ Info: hyperparameter optimization with EKI configured with Dict{Any, Any}("inflation" => 0.0001, "localization" => EnsembleKalmanProcesses.Localizers.NoLocalization(), "accelerator" => NesterovAccelerator{Float64}(Float64[], 1.0), "scheduler" => DataMisfitController{Float64, String}(Int64[], 20.0, "stop"), "cov_correction" => "shrinkage", "verbose" => true, "multithread" => "ensemble", "n_ensemble" => 40, "cov_sample_multiplier" => 1.0, "n_features_opt" => 40, "train_fraction" => 0.8, "n_cross_val_sets" => 2, "n_iteration" => 5)
└ @ CalibrateEmulateSample.Emulators /Users/julianschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/ScalarRandomFeature.jl:185
┌ Info: hyperparameter learning for 184 models using 478 training points, 120 validation points and 40 features
└ @ CalibrateEmulateSample.Emulators /Users/julianschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/ScalarRandomFeature.jl:379


estimate cov with 122 iterations...
estimate cov with 122 iterations...


┌ Info: Shrinkage scale: 0.03968010579133231, (0 = none, 1 = revert to scaled Identity)
│  shrinkage covariance condition number: 1853.5915221981975
└ @ CalibrateEmulateSample.Emulators /Users/julianschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/RandomFeature.jl:541
┌ Info: Shrinkage scale: 0.036079322065531584, (0 = none, 1 = revert to scaled Identity)
│  shrinkage covariance condition number: 1885.2837109028758
└ @ CalibrateEmulateSample.Emulators /Users/julianschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/RandomFeature.jl:541
┌ Warning: For 23 parameters, the recommended minimum ensemble size (`N_ens`) is 100. Got `N_ens` = 40`.
└ @ EnsembleKalmanProcesses /Users/julianschmitt/.julia/packages/EnsembleKalmanProcesses/71YBa/src/EnsembleKalmanProcess.jl:269


calculating 40 ensemble members...


...

4×4 Matrix{Any}:
 "name"                    …  "99% prior mass"
 "input_lowrank_diagonal"     (0.00012341, 8103.08)
 "input_lowrank_U"            (-300.0, 300.0)
 "sigma"                      (0.000872535, 44.0802)

4×4 Matrix{Any}:
 "name"                    …  "99% prior mass"
 "input_lowrank_diagonal"     (0.00012341, 8103.08)
 "input_lowrank_U"            (-300.0, 300.0)
 "sigma"                      (0.000872535, 44.0802)

4×4 Matrix{Any}:
 "name"                    …  "99% prior mass"
 "input_lowrank_diagonal"     (0.00012341, 8103.08)
 "input_lowrank_U"            (-300.0, 300.0)
 "sigma"                      (0.000872535, 44.0802)

4×4 Matrix{Any}:
 "name"                    …  "99% prior mass"
 "input_lowrank_diagonal"     (0.00012341, 8103.08)
 "input_lowrank_U"            (-300.0, 300.0)
 "sigma"                      (0.000872535, 44.0802)

Emulator{Float64}(ScalarRandomFeatureInterface{String, Random.TaskLocalRNG, SeparableKernel{LowRankFactor{Float64}, OneDimFactor}}(RandomFeatures.Methods.RandomFeatureMethod[RandomFeatures.Methods.RandomFeatureMethod{String, UniformScaling{Bool}}(RandomFeatures.Features.ScalarFeature{String, RandomFeatures.Features.Cosine}(100, RandomFeatures.Samplers.Sampler{Random.TaskLocalRNG}(ParameterDistribution with 2 entries: 
'xi' with EnsembleKalmanProcesses.ParameterDistributions.Constraint{EnsembleKalmanProcesses.ParameterDistributions.NoConstraint}[Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞), Bounds: (-∞, ∞)] over distribution EnsembleKalmanProcesses.ParameterDistributions.Parameterized(FullNormal( d

In [26]:
JLD2.save_object("emulator_rf_1.jld2", emulator_rf)

In [21]:
# sampling
init_sample = EKP.get_u_mean_final(eki_obj)

mcmc = MCMCWrapper(RWMHSampling(), get_obs(eki_obj)[1:index_max], prior, emulator_rf; init_params = init_sample);

new_step = optimize_stepsize(mcmc; init_stepsize = 0.1, N = 2000, discard_initial = 0);

chain = MarkovChainMonteCarlo.sample(mcmc, 100_000; stepsize = new_step, discard_initial = 2_000);

# Get posterior distribution from MCMC chain
posterior = get_posterior(mcmc, chain);

In [30]:
using CairoMakie

import EnsembleKalmanProcesses.Visualize as viz 

fig_pp = Figure(size = (1000, 1000))

viz.plot_parameter_distribution(fig_pp[1, 1], prior)
viz.plot_parameter_distribution!(fig_pp[1, 1], posterior)
fig_pp

UndefVarError: UndefVarError: `plot_parameter_distribution!` not defined in `EnsembleKalmanProcesses.Visualize`
Suggestion: check for spelling errors or missing imports.

In [32]:
using Plots
p = plot(prior)
plot!(p, posterior)
# save plot - use savefig() for Plots.jl
savefig(p, "prior_posterior_w_rf.png")

"/Users/julianschmitt/Documents/Research/loss/experiments/AtmosLossDesign/ces/prior_posterior_w_rf.png"

In [ ]:
get_outputs(cleaned_iopairs)
full_cov_matrix = Diagonal(vcat([diag(hcat(obs.covs...)) for obs in eki_obj.observation_series.observations]...))


nugget = 1e-3
overrides = Dict(
    "verbose" => true,
    # "scheduler" => DataMisfitController(terminate_at = 100.0),
    # "cov_sample_multiplier" => 1.0,
    # "n_iteration" => 8,
    # "n_features_opt" => 40,
)
n_features = 100
n_params = 21
kernel_structure = SeparableKernel(LowRankFactor(1, nugget), OneDimFactor())
mlt = ScalarRandomFeatureInterface(
    n_features,
    n_params,
    kernel_structure = kernel_structure,
    optimizer_options = overrides,
)


emulator_rf = Emulator(mlt, 
                    cleaned_iopairs;
                    obs_noise_cov = full_cov_matrix,
                    normalize_inputs = true,
                    retained_svd_frac = 0.95)

optimize_hyperparameters!(emulator_rf)


┌ Info: hyperparameter optimization with EKI configured with Dict{Any, Any}("inflation" => 0.0001, "localization" => EnsembleKalmanProcesses.Localizers.NoLocalization(), "accelerator" => EnsembleKalmanProcesses.NesterovAccelerator{Float64}(Float64[], 1.0), "scheduler" => EnsembleKalmanProcesses.DataMisfitController{Float64, String}(Int64[], 1000.0, "stop"), "cov_correction" => "shrinkage", "verbose" => true, "multithread" => "ensemble", "n_ensemble" => 100, "cov_sample_multiplier" => 10.0, "n_features_opt" => 100, "train_fraction" => 0.8, "n_cross_val_sets" => 2, "n_iteration" => 10)
└ @ CalibrateEmulateSample.Emulators /home/jschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/ScalarRandomFeature.jl:185


SVD truncated at k: 828/900


┌ Info: hyperparameter learning for 828 models using 381 training points, 96 validation points and 100 features
└ @ CalibrateEmulateSample.Emulators /home/jschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/ScalarRandomFeature.jl:379
